# 00 · Setup and smoke test (T4)
Run this once before any long runs. It checks that:
1. the DiaBlo layer and merge are numerically correct (unit tests and a tiny end-to-end pipeline),
2. a zero-initialised adapter leaves the base model's loss unchanged,
3. fp16 training on the real model runs with no NaNs, and save/kill/resume works,
4. **seconds per step**, used to finalise the time budget.

Runtime → Change runtime type → **T4 GPU**.

In [ ]:
# ── Common setup (run every session) ─────────────────────────────────────────
REPO_URL = "https://github.com/<YOUR_GITHUB_USER>/DiaBlo.git"   # your fork with the colab-repro branch
BRANCH   = "colab-repro"
MODEL    = "Qwen2.5-1.5B"                                       # short name or HF id
HF_ID    = "Qwen/Qwen2.5-1.5B"

import os, subprocess, torch
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/diablo"
os.makedirs(DRIVE, exist_ok=True)
DATA    = f"{DRIVE}/datasets/{MODEL}"     # tokenized data is tokenizer-specific
RESULTS = f"{DRIVE}/results"
EVALS   = f"{DRIVE}/eval"
os.environ["HF_HOME"] = f"{DRIVE}/hf_cache"   # keep downloads across sessions
os.environ["WANDB_MODE"] = "disabled"

REPO = "/content/DiaBlo"
if not os.path.exists(REPO):
    if os.path.exists(f"{DRIVE}/DiaBlo"):          # alternative: repo uploaded to Drive
        subprocess.run(["cp", "-r", f"{DRIVE}/DiaBlo", REPO], check=True)
    else:
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
%cd {REPO}
!git log -1 --oneline 2>/dev/null || true
!pip -q install peft
print(torch.__version__, torch.cuda.get_device_name(0), "bf16:", torch.cuda.is_bf16_supported())

In [ ]:
# Only needed for gated models (Llama). Add HF_TOKEN in Colab's "Secrets" (key icon) panel.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN set")
except Exception as e:
    print("No HF_TOKEN secret (fine for Qwen):", e)

## 1. Unit tests and tiny end-to-end pipeline (train → resume → merge → eval, DiaBlo and LoRA)

In [ ]:
!pip -q install pytest
!python -m pytest -q tests
!python tests/e2e_tiny.py --workdir /content/e2e --precision fp32

## 2. Tokenize GSM8k for this model (saved to Drive, run once per model)

In [ ]:
if not os.path.exists(f"{DATA}/GSM8k_processed/train"):
    !python data_processing/process_gsm8k.py --model_name {HF_ID} --output_dir {DATA} --max_length 512
from datasets import load_from_disk
tr = load_from_disk(f"{DATA}/GSM8k_processed/train")
lens = [len(x) for x in tr["input_ids"]]
print(len(tr), "train examples | max len", max(lens), "| mean", sum(lens)/len(lens))

## 3. Zero-init check: step-0 loss with adapters == base loss

In [ ]:
import torch, copy
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch.nn.utils.rnn import pad_sequence
from diablo import setup_pad_token, apply_adapter, parse_target_modules, mark_trainable

tok = AutoTokenizer.from_pretrained(HF_ID)
base = AutoModelForCausalLM.from_pretrained(HF_ID, torch_dtype=torch.float16).cuda().eval()
setup_pad_token(tok, base)
b = [tr[i] for i in range(4)]
pad = lambda key, v: torch.swapaxes(pad_sequence([torch.tensor(x[key]) for x in b], padding_value=v, padding_side="left"), 0, 1).cuda()
batch = dict(input_ids=pad("input_ids", tok.pad_token_id), attention_mask=pad("attention_mask", 0), labels=pad("labels", -100))
with torch.no_grad():
    l0 = base(**batch).loss.item()
    m = apply_adapter(base, "diablo", parse_target_modules("qkvud"), num_blocks=64, dropout=0.0)
    m.cuda().half().eval()
    tot, trn = mark_trainable(m)
    with torch.autocast("cuda", dtype=torch.float16):
        l1 = m(**batch).loss.item()
print(f"base loss {l0:.5f} | DiaBlo(zero-init) loss {l1:.5f} | trainable {trn/1e6:.2f}M")
assert abs(l0 - l1) < 1e-2, "zero-init adapter changed the model output"
del base, m; torch.cuda.empty_cache()

## 4. Real-model fp16 smoke run: 20 steps, then resume to 30. Measures s/step

In [ ]:
import json, time
SMOKE = f"{DRIVE}/smoke"
!rm -rf {SMOKE}
common = f"--model_name {MODEL} --task GSM8k --data_dir {DATA} --output_dir {SMOKE} --precision fp16 --per_device_batch_size 4 --gradient_accumulation_steps 4 --save_steps 10 --warmup_steps 5"
t = time.time()
!python train.py {common} --method diablo --num_blocks 64 --lr 5e-4 --run_name smoke_diablo --max_steps 20
t_diablo = (time.time() - t)
!python train.py {common} --method diablo --num_blocks 64 --lr 5e-4 --run_name smoke_diablo --max_steps 30 --resume

In [ ]:
st = json.load(open(f"{SMOKE}/smoke_diablo/checkpoint-30/trainer_state.json"))
losses = [h["loss"] for h in st["log_history"] if "loss" in h]
print("losses every 10 steps:", losses)
assert all(l == l for l in losses), "NaN loss -> retry with --precision fp32 --gradient_checkpointing"
# train_runtime of the resumed run covers exactly steps 21-30 (no model-loading overhead)
lh = json.load(open(f"{SMOKE}/smoke_diablo/log_history.json"))
sps = lh[-1]["train_runtime"] / 10
print(f"~{sps:.1f} s/step  ->  full run (1401 steps) ~ {1401*sps/3600:.1f} h")

In [ ]:
# Same check for the LoRA baseline (r=16 ~ DiaBlo N=64 in params)
t = time.time()
!python train.py {common} --method lora --lora_r 16 --lr 2e-4 --run_name smoke_lora --max_steps 20
print(f"LoRA ~{(time.time()-t)/20:.1f} s/step")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

**If you get OOM:** use `--per_device_batch_size 2 --gradient_accumulation_steps 8`, which keeps the effective batch at 16 and the results comparable. You can also add `--gradient_checkpointing`.
**If you get NaN losses:** use `--precision fp32 --gradient_checkpointing` for **all** runs, so every method is trained the same way.